### TITLE

##### House Credit Default Prediction

This project predicts whether a loan applicant will default using the Home Credit dataset.

##### Objectives
- Clean and preprocess financial data
- Handle missing values and categorical features
- Train multiple machine learning models
- Compare models using ROC-AUC
- Select the best performing model

### LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, precision_recall_curve

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

### DATA HANDLING

In [ ]:
df = pd.read_csv("data/application_train.csv")
df.head()

missing_col = df.isnull().mean()
df = df.drop(columns=missing_col[missing_col > 0.50].index)
n_df = df.drop("SK_ID_CURR", axis=1)

numeric_cols = n_df.select_dtypes(include=['number']).columns
object_cols = n_df.select_dtypes(include=['object']).columns
n_df[numeric_cols] = n_df[numeric_cols].fillna(n_df[numeric_cols].median())
n_df[object_cols] = n_df[object_cols].fillna("Missing")

encoder = LabelEncoder()
for col in object_cols:
    n_df[col] = encoder.fit_transform(n_df[col])

### TRAIN TEST SPLIT

In [ ]:
X = n_df.drop(columns="TARGET")
y = n_df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Baseline Model (Random Forest)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

y_prob = rf_model.predict_proba(X_test)[:,1]
y_pred = (y_prob > 0.23).astype(int)

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

### Model Comparison

In [ ]:
### LOGISTIC REGRESSION
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

lr_model.fit(X_train_scaled, y_train)

y_prob_lr = lr_model.predict_proba(X_test_scaled)[:,1]

print("ROC AUC:", roc_auc_score(y_test, y_prob_lr))

### DECISION TREE
model_dt = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

model_dt.fit(X_train, y_train)

y_prob_dt = model_dt.predict_proba(X_test)[:,1]
y_pred_dt = (y_prob_dt > 0.69).astype(int)

print("Decision Tree")
print(classification_report(y_test, y_pred_dt))
print("ROC AUC:", roc_auc_score(y_test, y_prob_dt))

### K-Nearest Neighbors
model_knn = KNeighborsClassifier(n_neighbors=5)

model_knn.fit(X_train_scaled, y_train)

y_prob_knn = model_knn.predict_proba(X_test_scaled)[:,1]
y_pred_knn = (y_prob_knn > 0.2).astype(int)

print("KNN")
print(classification_report(y_test, y_pred_knn))
print("ROC AUC:", roc_auc_score(y_test, y_prob_knn))

### GRADIENT BOOSTING
model_gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model_gb.fit(X_train, y_train)

y_prob_gb = model_gb.predict_proba(X_test)[:,1]
y_pred_gb = (y_prob_gb > 0.11).astype(int)

print("Gradient Boosting")
print(classification_report(y_test, y_pred_gb))
print("ROC AUC:", roc_auc_score(y_test, y_prob_gb))

### LIGHTBGM
class_imb=(y_train==0).sum()/(y_train==1).sum()
model_xg=xg(n_estimators=400,learning_rate=0.05,scale_pos_weight=class_imb,max_depth=3,min_child_weight=1,random_state=42)
model_xg.fit(x_train,y_train)
y_prob=model_xg.predict_proba(x_test)[:,1]
y_pred=model_xg.predict(x_test)
y_new=(y_prob>0.60).astype(int)
print(accuracy_score(y_test,y_new))
print(classification_report(y_test,y_new))
print(roc_auc_score(y_test,y_prob))

### Cross Validation

In [ ]:
sk = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=sk,
    scoring="roc_auc"
)

print("CV Scores:", scores)
print("Mean:", scores.mean())

### HYPERPARAMETER TUNING

In [ ]:
param_grid = {
    "n_estimators":[50,100],
    "max_depth":[5,10,None],
    "min_samples_split":[2,5]
}

grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced"),
    param_grid,
    cv=sk,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X,y)

print(grid.best_params_)
print(grid.best_score_)

### FEATURE ENGINEERING

In [ ]:
n_df["ext_source_mean"] = (n_df["EXT_SOURCE_2"] + n_df["EXT_SOURCE_3"]) / 2
n_df["ext_source_pred"] = n_df["EXT_SOURCE_2"] * n_df["EXT_SOURCE_3"]
n_df["ext_diff"] = n_df["EXT_SOURCE_2"] - n_df["EXT_SOURCE_3"]
n_df["credit_income_ratio"] = n_df["AMT_CREDIT"] / n_df["AMT_INCOME_TOTAL"]
n_df["annuity_income_ratio"] = n_df["AMT_ANNUITY"] / n_df["AMT_INCOME_TOTAL"]
n_df["credit_annuity_ratio"] = n_df["AMT_CREDIT"] / n_df["AMT_ANNUITY"]

### Final Model (XGBoost)

In [ ]:
class_imb = (y_train==0).sum()/(y_train==1).sum()

model_xg = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    scale_pos_weight=class_imb,
    max_depth=3
)

model_xg.fit(X_train,y_train)

y_prob = model_xg.predict_proba(X_test)[:,1]

print("ROC AUC:", roc_auc_score(y_test,y_prob))

### THRESHOLD OPTIMIZATION

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

f1 = 2 * precision * recall / (precision + recall + 1e-9)

best_idx = np.argmax(f1)

print("Best threshold:", thresholds[best_idx])
print("Best F1:", f1[best_idx])

### 